In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ClassificationReport,
    RocCurveDisplay,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Load feature set
df = pd.read_csv("donor_features.csv")

# 2. Define Binary Target: Is the donor active in the last 90 days?
df["is_active_90d"] = (df["donations_last_90_days"] > 0).astype(int)

# 3. Separate Features (X) and Target (y)
# Drop target and features that directly leak the target
X = df[
    [
        "donation_count",
        "total_donations",
        "average_donation",
        "campaign_count",
    ]
]
y = df["is_active_90d"]

# 4. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Target Distribution:\n{y.value_counts(normalize=True).round(3)}")

In [ ]:
# Create pipelines for comparison
pipelines = {
    "Baseline Logistic Regression": Pipeline(
        [("scaler", StandardScaler()), ("clf", LogisticRegression())]
    ),
    "Balanced Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(class_weight="balanced")),
        ]
    ),
    "Random Forest (Benchmark)": Pipeline(
        [("clf", RandomForestClassifier(n_estimators=100, random_state=42))]
    ),
}

# Hyperparameter Grid Search for Logistic Regression tuning
param_grid_lr = {
    "clf__C": [0.01, 0.1, 1.0, 10.0],
    "clf__penalty": ["l2"],
    "clf__solver": ["lbfgs"],
}

grid_lr = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression())]),
    param_grid_lr,
    cv=5,
    scoring="roc_auc",
)
grid_lr.fit(X_train, y_train)

# Add tuned model to pipelines
pipelines["Tuned Logistic Regression"] = grid_lr.best_estimator_

print(f"Best Logistic Regression Params: {grid_lr.best_params_}")

In [ ]:
results = []
plt.figure(figsize=(9, 6))

for name, model in pipelines.items():
    if name != "Tuned Logistic Regression":
        model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Metrics
    auc = roc_auc_score(y_test, y_proba)
    results.append(
        {
            "Model": name,
            "ROC-AUC": round(auc, 4),
        }
    )

    # Plot ROC Curves
    RocCurveDisplay.from_predictions(
        y_test, y_proba, name=f"{name} (AUC = {auc:.3f})", ax=plt.gca()
    )

plt.plot([0, 1], [0, 1], "k--", label="Chance Level")
plt.title("ROC-AUC Model Comparison", fontsize=14, fontweight="bold")
plt.legend(loc="lower right")
plt.show()

# Display Summary Metrics Table
df_results = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False)
print(df_results.to_string(index=False))

In [ ]:
# Extract coefficients from the tuned model
tuned_model = pipelines["Tuned Logistic Regression"].named_steps["clf"]
coefficients = tuned_model.coef_[0]

df_coef = pd.DataFrame(
    {
        "Feature": X.columns,
        "Coefficient": coefficients,
        "Odds Ratio": np.exp(coefficients),
    }
).sort_values(by="Odds Ratio", ascending=False)

print("\nFeature Impact (Odds Ratios):")
print(df_coef.to_string(index=False))